# Can local LLMs write CI configurations? Gemma 3 12B, Llama 3.1 8B and CodeGemma 7B

This notebook is part of a study on generating CI configurations with LLMs.
It gives 10 plain-English pipeline descriptions to three open models, **gemma3:12b, llama3.1:8b and codegemma:7b**,
and saves every generated file so it can be graded by hand.

The same 10 descriptions were also given to qwen2.5-coder:7b, qwen2.5-coder:14b and deepseek-coder-v2:16b
(notebooks in `azure/notebook/` and `jenkins/notebook/`), so all six models can be compared.

### The question
Can a small model that runs on one GPU write a **correct** CI configuration
when it is given only a plain-English description of that file?

### What "CI configuration" means here
The pipeline definition files of CI services: Azure Pipelines, GitHub Actions and Jenkins.
These files define how a project is built and tested and, in some projects, also how it is
packaged and deployed. As in Ghaleb & Rathnayake (ICSME 2025), the whole file counts as a CI configuration.
3 of the 10 configurations only build and test (Azure 1–3); the other 7 also deliver or deploy.
Section 3 marks the scope of each one.

### How the study works
```
real CI configuration on GitHub ─▶  description written from it  ─▶  description checked against the file
                                                                              │
         grade by hand  ◀──  generated file  ◀──  3 models (same prompt, temperature 0, seed 42)
```
1. **10 real projects** (5 Azure, 5 Jenkins) that keep their own CI configuration file in their repo.
2. A **description** of each file, listing every trigger, agent, step, command and value.
3. Each description was **checked claim by claim** against its file (Azure 121 checks, Jenkins 75 checks, all pass),
   so a model is never blamed for a vague or wrong description.
4. **This notebook:** each model writes a file from each description. The model never sees the original.
5. **After this notebook:** each generated file is compared with the original and given a remark
   (see [Section 9](#9.-Next-step:-grading-the-outputs)).

### What is kept the same as the qwen / deepseek notebooks
Only the models change. Everything else is copied exactly from those notebooks:
the 10 descriptions, both system prompts, `temperature = 0`, `seed = 42`, `num_ctx = 8192`,
`num_predict = 2048`, the code-extraction functions and the validity checks.
That is what makes all six models comparable.

### Contents
1. [Setup: install and start Ollama](#1.-Setup:-install-and-start-Ollama)
2. [Choose and download the models](#2.-Choose-and-download-the-models)
3. [The 10 descriptions](#3.-The-10-descriptions)
4. [The prompts and generation settings](#4.-The-prompts-and-generation-settings)
5. [Pulling the code out of the reply, and quick checks](#5.-Pulling-the-code-out-of-the-reply,-and-quick-checks)
6. [Generate all 30 files](#6.-Generate-all-30-files)
7. [Save the results](#7.-Save-the-results)
8. [Print every generated file](#8.-Print-every-generated-file)
9. [Next step: grading the outputs](#9.-Next-step:-grading-the-outputs)
10. [Troubleshooting](#10.-Troubleshooting)

### How to run it on Kaggle
1. **File → Import Notebook** and choose this file.
2. Right panel → **Settings**: **Accelerator: GPU T4 x2**, **Internet: On**
   (Kaggle only allows internet on a phone-verified account).
3. **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers even if you close the tab.
   It takes about **15–25 minutes**, mostly downloading the models (about 18 GB in total).
4. Open the saved version's **Output** tab and download `free_models_outputs.zip`.

No API key is needed: all three models run locally on Kaggle's GPU.

## 1. Setup: install and start Ollama

**What this does:** installs [Ollama](https://ollama.com), the program that downloads open models and runs them
on the GPU, then starts it in the background as a small local server at `127.0.0.1:11434`.
The rest of the notebook sends each description to that server and reads back the reply.

**What you should see:** the GPU name (for example `Tesla T4`) and then `Ollama server is up`.
If it says `NO GPU`, turn on the accelerator in Settings. If Ollama doesn't start, check that
**Internet is On** (see [Troubleshooting](#10.-Troubleshooting)).

In [ ]:
# System packages: zstd is needed to unpack Ollama; then the official Ollama install script.
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip -q install pyyaml

import os, re, json, time, shutil, subprocess, requests
import yaml
import pandas as pd

# Which GPU is this session using? (Ollama falls back to the CPU if there is none)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "NO GPU - turn on the accelerator in Settings")

WORK = "/kaggle/working"          # everything saved here appears in the Output tab
OLLAMA = "http://127.0.0.1:11434"

# Start the Ollama server in the background and log to ollama.log
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
server = subprocess.Popen(["ollama", "serve"], stdout=open(f"{WORK}/ollama.log", "w"),
                          stderr=subprocess.STDOUT)

# Wait up to 60 seconds for the server to answer
for _ in range(60):
    try:
        requests.get(OLLAMA, timeout=2); print("Ollama server is up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())


def ollama_log_tail(n=25):
    """Last lines of Ollama's own log: this is where the real reason for an error is."""
    try:
        return "".join(open(f"{WORK}/ollama.log").readlines()[-n:])
    except Exception:
        return "(no ollama.log)"

## 2. Choose and download the models

**What this does:** lists the three models and downloads each one from the Ollama library.

| Folder name | Ollama tag | Size | Kind | Why it is in the study |
|---|---|---|---|---|
| `gemma3-12b` | `gemma3:12b` | about 8 GB | general-purpose (Google) | best open model in Ghaleb & Rathnayake (ICSME 2025) |
| `llama3.1-8b` | `llama3.1:8b` | about 5 GB | general-purpose (Meta) | a second model family of similar size |
| `codegemma-7b` | `codegemma:7b` | about 5 GB | code-specialised (Google) | tests whether code training helps with CI files |

Together with the qwen and deepseek models this gives 6 open models: 4 code-specialised
(qwen 7b, qwen 14b, deepseek 16b, codegemma 7b) and 2 general-purpose (gemma 12b, llama 8b).

The folder name is used for the output folders, so the files sit next to the qwen and deepseek
folders in the repo's `azure/generated/` and `jenkins/generated/`.

**What you should see:** `ok` after each model, then a table from `ollama list` with all three models.
If a download stops partway, just run this cell again: Ollama resumes where it stopped.

In [ ]:
# folder name -> Ollama tag
MODELS = {
    "gemma3-12b":   "gemma3:12b",
    "llama3.1-8b":  "llama3.1:8b",
    "codegemma-7b": "codegemma:7b",
}

for name, tag in MODELS.items():
    print("pulling", tag, "...")
    r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else r.stderr[-500:])

print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

### 2b. Test that each model loads and answers

**What this does:** asks each model to reply with one word, before the real run. This catches a model that
downloaded fine but can't be loaded (for example, not enough GPU memory or a GPU Ollama doesn't support),
and shows Ollama's own error message if that happens.

**What you should see:** `answers` and the GPU memory used for each model. A model that fails here is skipped
in Section 6, and the other models still run.

In [ ]:
WORKING_MODELS = {}
for name, tag in MODELS.items():
    r = requests.post(f"{OLLAMA}/api/generate", json={"model": tag, "prompt": "Reply with the word ok.",
                      "stream": False, "options": {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 5}}, timeout=1800)
    if r.ok:
        ps = requests.get(f"{OLLAMA}/api/ps").json().get("models", [])
        vram = sum(m.get("size_vram", 0) for m in ps) / 1e9
        print(f"{name:13s} answers: {r.json()['response'].strip()[:20]!r}   on GPU: {vram:.1f} GB")
        WORKING_MODELS[name] = tag
    else:
        print(f"{name:13s} FAILED: HTTP {r.status_code}: {r.text[:300]}")
        print("---- last lines of ollama.log ----")
        print(ollama_log_tail())
    requests.post(f"{OLLAMA}/api/generate", json={"model": tag, "keep_alive": 0})   # unload

print("\nModels that will run:", list(WORKING_MODELS) or "none - see the errors above")

## 3. The 10 descriptions

**What this does:** stores the 10 descriptions the models will work from. They are **the same descriptions given to the qwen and deepseek models**.
Each one was written from a real CI configuration file and checked claim by claim against it.
The original files are not copied here; `repo` links to each project, and `descriptions.md` in the repo
links to the exact file and commit that was used.

### Azure (5 YAML files)
| # | Project | Type | Scope | Output file |
|---|---|---|---|---|
| 1 | Azure Pipeline Basic (Python + pytest) | Azure Pipelines | Build & test | `1_python_pytest_azp.yml` |
| 2 | JavaScript E2E Express Server (Node.js + JUnit test results) | Azure Pipelines | Build & test | `2_node_junit_azp.yml` |
| 3 | Azure Batch Samples (Python, Java, .NET) | Azure Pipelines | Build & test | `3_batch_multilang_azp.yml` |
| 4 | Todo App: Node.js + MongoDB on Azure (azd deploy) | GitHub Actions (deploys to Azure) | Build & test + deploy | `4_todo_azd_gha.yml` |
| 5 | Java EE 7 Petstore: Deploy to Azure Web App | GitHub Actions (deploys to Azure) | Build & test + deploy | `5_petstore_webapp_gha.yml` |

### Jenkins (5 Groovy Jenkinsfiles)
| # | Project | Type | Scope | Output file |
|---|---|---|---|---|
| 1 | Simple Java Maven App | Declarative pipeline | Build & test + deliver | `1_java_maven.groovy` |
| 2 | Simple Python PyInstaller App | Declarative pipeline | Build & test + deliver | `2_python_pyinstaller.groovy` |
| 3 | Multibranch Pipeline Project (Node.js / React) | Declarative pipeline | Build & test + deploy | `3_multibranch_node.groovy` |
| 4 | DevOps Spring Boot (Maven + Docker Hub) | Declarative pipeline | Build & test + deliver | `4_springboot_docker.groovy` |
| 5 | Spring Boot Docker Deploy (Scripted Pipeline) | Scripted pipeline | Build & test + deploy | `5_scripted_docker_deploy.groovy` |

**Scope** shows whether the configuration only builds and tests, or also delivers (publishes an artifact or image)
or deploys (to Azure or a running container). Report results for the two groups separately.

**`slug`** is the output file name, the same as in the repo, so the files of all six models can be compared side by side.

**What you should see:** the 10 slugs with the word count of each description.

In [ ]:
# Azure Pipelines and GitHub Actions (deploying to Azure): 5 descriptions
AZURE_ITEMS = [
    {
        'no': 1,
        'slug': '1_python_pytest_azp',
        'title': 'Azure Pipeline Basic (Python + pytest)',
        'repo': 'https://github.com/atulkamble/azure-pipeline-basic',
        'description': "Create an Azure Pipelines YAML file (azure-pipelines.yml) for a Python project. The pipeline is triggered by commits to the main branch and runs on a Microsoft-hosted ubuntu-latest agent. Define a pipeline variable named pythonVersion with the value '3.11'. The pipeline has three steps: (1) the UsePythonVersion@0 task, selecting the Python version from the pythonVersion variable; (2) a script step with display name 'Install dependencies' that upgrades pip and installs the packages listed in requirements.txt; (3) a script step with display name 'Run tests' that runs pytest in quiet mode (pytest -q).",
    },
    {
        'no': 2,
        'slug': '2_node_junit_azp',
        'title': 'JavaScript E2E Express Server (Node.js + JUnit test results)',
        'repo': 'https://github.com/Azure-Samples/js-e2e-express-server',
        'description': "Create an Azure Pipelines YAML file for a Node.js Express server project. The pipeline is triggered by commits to the main branch and runs on a Microsoft-hosted ubuntu-latest agent. It has three steps: (1) the NodeTool@0 task installing Node.js version '10.x', with display name 'Install Node.js'; (2) a script step (no display name) that runs npm install, then npm run build, then npm test; (3) the PublishTestResults@2 task that publishes test results in JUnit format from files matching '**/TEST-*.xml'. This publish step has the display name 'npm install and build'.",
    },
    {
        'no': 3,
        'slug': '3_batch_multilang_azp',
        'title': 'Azure Batch Samples (Python, Java, .NET)',
        'repo': 'https://github.com/Azure-Samples/azure-batch-samples',
        'description': "Create an Azure Pipelines YAML file that tests the Azure Batch sample code in three languages. The pipeline is triggered by commits to the master branch. Define three independent jobs, each running on a Microsoft-hosted windows-2019 agent. Job 'PythonTest' (display name 'Run lint tests') uses a matrix strategy with two configurations: PythonMinVersion with python.version '3.7' and PythonMaxVersion with python.version '3.10'. Its steps are: the UsePythonVersion@0 task with versionSpec '$(python.version)', then one script that upgrades setuptools, installs requirements from $(System.DefaultWorkingDirectory)/Python/Batch/requirements.txt, installs coveralls, flake8, pytest, pytest-cov and requests_mock, and runs flake8 on $(System.DefaultWorkingDirectory)/Python/Batch. Job 'JavaTest' has one Maven@3 task using the POM file $(System.DefaultWorkingDirectory)/Java/pom.xml, mavenOptions '-Xmx3072m', javaHomeOption 'JDKVersion', jdkVersionOption '1.8', jdkArchitectureOption 'x64', publishJUnitResults false, testResultsFiles '**/TEST-*.xml' and goals 'package'. Job 'DotNetTest' has two DotNetCoreCLI@2 tasks: the first runs the 'restore' command on '**/*.csproj' with feedsToUse 'select', the second runs the 'build' command on '**/*.csproj'.",
    },
    {
        'no': 4,
        'slug': '4_todo_azd_gha',
        'title': 'Todo App: Node.js + MongoDB on Azure (azd deploy)',
        'repo': 'https://github.com/Azure-Samples/todo-nodejs-mongo',
        'description': 'Create a GitHub Actions workflow that deploys a Node.js + MongoDB todo application to Azure using the Azure Developer CLI (azd). The workflow runs on manual dispatch (workflow_dispatch) and on pushes to the main or master branch. Grant the permissions id-token: write and contents: read so it can log in with OpenID Connect federated credentials. It has one job named \'build\' on ubuntu-latest with job-level environment variables AZURE_CLIENT_ID, AZURE_TENANT_ID and AZURE_SUBSCRIPTION_ID taken from repository variables (vars) and AZURE_CREDENTIALS taken from secrets. The steps are: \'Checkout\' using actions/checkout@v4; \'Install azd\' using Azure/setup-azd@v2; \'Install Nodejs\' using actions/setup-node@v4 with node-version 18; \'Log in with Azure (Federated Credentials)\', which runs only if AZURE_CLIENT_ID is not empty and runs azd auth login with the client ID, --federated-credential-provider "github" and the tenant ID in a pwsh shell; \'Log in with Azure (Client Credentials)\', which runs only if AZURE_CREDENTIALS is not empty, parses the AZURE_CREDENTIALS JSON in pwsh, masks the client secret, and runs azd auth login with the client ID, client secret and tenant ID; \'Provision Infrastructure\', which runs azd provision --no-prompt with environment variables AZURE_ENV_NAME, AZURE_LOCATION and AZURE_SUBSCRIPTION_ID from vars and AZD_INITIAL_ENVIRONMENT_CONFIG from secrets; \'Deploy Application\', which runs azd deploy --no-prompt with AZURE_ENV_NAME, AZURE_LOCATION and AZURE_SUBSCRIPTION_ID from vars.',
    },
    {
        'no': 5,
        'slug': '5_petstore_webapp_gha',
        'title': 'Java EE 7 Petstore: Deploy to Azure Web App',
        'repo': 'https://github.com/Azure-Samples/Java-application-petstore-ee7',
        'description': "Create a GitHub Actions workflow named 'Deploy Java Web App to Azure' that builds a Java EE 7 petstore application with Maven and deploys the WAR file to Azure App Service. It runs on push and on pull_request. Define workflow-level environment variables: AZURE_WEBAPP_NAME = JavaPetstore; AZURE_WEBAPP_PACKAGE_PATH = ${{ github.workspace }}; JAVA_VERSION = '1.8'; AZURE_WEBAPP_PUBLISH_PROFILE from the secret AZURE_WEBAPP_PUBLISH_PROFILE; CONTAINER_REGISTRY = ${{ secrets.REGISTRY_USERNAME }}.azurecr.io; AZURE_RESOURCE_GROUP = ActionsDemo; AZURE_APP_PLAN = ActionPlan. It has one job, 'build-and-deploy', on ubuntu-latest with these steps: check out the code with actions/checkout@master; 'Set up JDK 1.8' using actions/setup-java@v1 with java-version from JAVA_VERSION; 'Build with Maven' running mvn clean install --file pom.xml -Dmaven.test.skip=true; 'Azure authentication' using azure/login@v1 with creds from the secret AZURE_CREDENTIALS; 'Deploy to Azure WebApp' using azure/webapps-deploy@v2 with app-name from AZURE_WEBAPP_NAME and package '${{ env.AZURE_WEBAPP_PACKAGE_PATH }}/target/*.war'.",
    },
]

# Jenkins (Groovy Jenkinsfiles): 4 declarative pipelines and 1 scripted pipeline
JENKINS_ITEMS = [
    {
        'no': 1,
        'slug': '1_java_maven',
        'title': 'Simple Java Maven App',
        'repo': 'https://github.com/jenkins-docs/simple-java-maven-app',
        'type': 'Declarative pipeline',
        'description': "Write a Jenkins declarative pipeline (Jenkinsfile) for a Java Maven project. It runs on any agent whose label is not 'windows' (label '!windows') and uses the option skipStagesAfterUnstable(). It has three stages: 'Build', which runs the shell command mvn -B -DskipTests clean package; 'Test', which runs mvn test and, in a post 'always' block, publishes the JUnit test reports from target/surefire-reports/*.xml; 'Deliver', which runs the shell script ./jenkins/scripts/deliver.sh.",
    },
    {
        'no': 2,
        'slug': '2_python_pyinstaller',
        'title': 'Simple Python PyInstaller App',
        'repo': 'https://github.com/jenkins-docs/simple-python-pyinstaller-app',
        'type': 'Declarative pipeline',
        'description': "Write a Jenkins declarative pipeline (Jenkinsfile) for a small Python command-line app. It runs on any agent and uses the option skipStagesAfterUnstable(). It has three stages: 'Build', which byte-compiles the sources with python3 -m py_compile sources/add2vals.py sources/calc.py; 'Test', which runs py.test --verbose --junit-xml test-reports/results.xml sources/test_calc.py and, in a post 'always' block, publishes the JUnit results from test-reports/results.xml; 'Deliver', which builds a single-file executable with pyinstaller --onefile sources/add2vals.py and, in a post 'success' block, archives the artifact dist/add2vals.",
    },
    {
        'no': 3,
        'slug': '3_multibranch_node',
        'title': 'Multibranch Pipeline Project (Node.js / React)',
        'repo': 'https://github.com/jenkins-docs/building-a-multibranch-pipeline-project',
        'type': 'Declarative pipeline',
        'description': 'Write a Jenkins declarative pipeline (Jenkinsfile) for a Node.js/React app built as a multibranch pipeline. The whole pipeline runs inside a Docker agent using the image \'node:lts-alpine\' with the args \'-p 3000:3000 -p 5000:5000\'. Set the environment variable CI = \'true\'. It has four stages: \'Build\', which runs npm install; \'Test\', which runs ./jenkins/scripts/test.sh; \'Deliver for development\', which runs only when the branch is \'development\': it runs ./jenkins/scripts/deliver-for-development.sh, then pauses with an input step whose message is \'Finished using the web site? (Click "Proceed" to continue)\', then runs ./jenkins/scripts/kill.sh; \'Deploy for production\', which runs only when the branch is \'production\': it runs ./jenkins/scripts/deploy-for-production.sh, then the same input message, then ./jenkins/scripts/kill.sh.',
    },
    {
        'no': 4,
        'slug': '4_springboot_docker',
        'title': 'DevOps Spring Boot (Maven + Docker Hub)',
        'repo': 'https://github.com/saravanan81java/devops-springboot',
        'type': 'Declarative pipeline',
        'description': 'Write a Jenkins declarative pipeline (Jenkinsfile) that builds a Spring Boot app with Maven and pushes a Docker image to Docker Hub. It runs on any agent and uses the Maven tool installation named \'Default\'. Set the environment variable PATH to "/usr/local/bin:$PATH" so Docker can be found. It has three stages: \'Build Maven\', which checks out the code with checkout(scmGit(...)) from the main branch (*/main) of https://github.com/saravanan81java/devops-springboot using the credentials ID \'saravanan81java\' and no extensions, then runs mvn clean install; \'Build docker image\', which inside a script block runs docker build -t saran81/devops-springboot-integration . ; \'Push image to Hub\', which inside a script block uses withCredentials with a secret text credential \'dockerhubpwd\' bound to the variable dockerhubpwd to run docker login -u saran81 -p ${dockerhubpwd}, and then runs docker push saran81/devops-springboot-integration.',
    },
    {
        'no': 5,
        'slug': '5_scripted_docker_deploy',
        'title': 'Spring Boot Docker Deploy (Scripted Pipeline)',
        'repo': 'https://github.com/vdharmaraj/PGDO_Proj3',
        'type': 'Scripted pipeline',
        'description': 'Write a Jenkins scripted pipeline (Jenkinsfile using node { ... }, not the declarative pipeline block) that builds a Spring Boot app, builds and pushes a Docker image, and redeploys it as a container. Inside node, define the variables application = "devopsexample", dockerhubaccountid = "vikidvg", mvnHome = tool \'maven-3.5.2\', an empty dockerImage, and dockerImageTag = "${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}". The stages are: \'Clone Repo\', which runs git with url \'https://github.com/vdharmaraj/PGDO_Proj3.git\' and branch \'main\', then sets mvnHome = tool \'maven-3.5.2\' again; \'Build Project\', which runs "\'${mvnHome}/bin/mvn\' clean install"; \'Build Docker Image with new code\', which sets dockerImage = docker.build("${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}"); \'Push Image to Remote Repo\', which echoes "Docker Image Tag Name ---> ${dockerImageTag}" and then, inside docker.withRegistry(\'\', \'dockerHub\'), pushes the image with the tag "${env.BUILD_NUMBER}" and with the tag "latest"; \'Remove running container with old code\', which runs docker rm -f \\$(docker ps -a -f name=devopsexample -q) || true; \'Deploy Docker Image with new changes\', which runs docker run --name devopsexample -d -p 2222:2222 ${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}; \'Remove old images\', which runs docker rmi ${dockerhubaccountid}/${application}:latest -f.',
    },
]

for it in AZURE_ITEMS + JENKINS_ITEMS:
    print(f"{it['slug']:28s} {len(it['description'].split()):4d} words")

## 4. The prompts and generation settings

**What this does:** defines what every model is told, and how it should generate. All of it is **copied from the qwen / deepseek notebooks**.

**System prompts.** There is one per platform. Each tells the model to follow the description exactly
(names, versions, paths and values), add nothing it doesn't ask for, leave nothing out, and reply with only the code
inside one code block. The model gets this instruction plus the description, and **never sees the original file**.

**Settings**

| Setting | Value | Meaning |
|---|---|---|
| `temperature` | 0 | always pick the most likely next word, so output is as repeatable as possible |
| `seed` | 42 | fixed random seed, for the same reason |
| `num_ctx` | 8192 | the model can read up to 8192 tokens (the longest prompt is far shorter) |
| `num_predict` | 2048 | the model may write up to 2048 tokens (enough for the longest original file) |

A note for the write-up: temperature 0 with a fixed seed makes re-runs **very close**, but Ollama does not
guarantee identical output across different GPUs or Ollama versions. That's why every reply is saved.

In [ ]:
# Instruction for the 5 Azure tasks
AZURE_SYSTEM = 'You are an experienced DevOps engineer. Write the CI/CD YAML file described by the user. Follow the description exactly: use the exact names, versions, paths and values it gives, do not add anything it does not ask for, and do not leave anything out. Output only the YAML inside a single ```yaml code block, with no explanation.'

# Instruction for the 5 Jenkins tasks
JENKINS_SYSTEM = 'You are an experienced DevOps engineer. Write the Jenkinsfile (Jenkins pipeline in Groovy) described by the user. Follow the description exactly: use the exact stage names, commands, paths, credential IDs and values it gives, use the pipeline style it asks for (declarative or scripted), do not add anything it does not ask for, and do not leave anything out. Output only the Jenkinsfile inside a single ```groovy code block, with no explanation.'

# Generation settings, identical for every model and every task
OPTIONS = {'temperature': 0, 'seed': 42, 'num_ctx': 8192, 'num_predict': 2048}

print("Azure prompt:  ", AZURE_SYSTEM[:90], "...")
print("Jenkins prompt:", JENKINS_SYSTEM[:90], "...")
print("Options:       ", OPTIONS)

## 5. Pulling the code out of the reply, and quick checks

**What this does:** defines four small helper functions, unchanged from the qwen / deepseek notebooks.

| Function | Used for | What it does |
|---|---|---|
| `extract_yaml` | Azure | takes the YAML out of the model's reply: the longest ```` ```yaml ```` block, or the whole reply if there is no block |
| `parses` | Azure | checks the YAML can be read by a YAML parser: `yes`, or `no:` and the error |
| `extract_code` | Jenkins | the same as `extract_yaml`, for ```` ```groovy ```` blocks |
| `basic_check` | Jenkins | a quick sanity check, **not** a real Groovy compile: balanced `{}` `()` `[]` (ignoring strings and comments) and the file starts with `pipeline {` or `node` |

**Why the checks are not enough.** For the qwen and deepseek models, all 15 Azure files parsed and all 15 Jenkinsfiles passed
`basic_check`, yet several would fail if run. A passing check only means the file is not broken on the surface.
Correctness is decided by the manual grading in [Section 9](#9.-Next-step:-grading-the-outputs).

In [ ]:
# --- Azure: take the YAML out of the reply, then check it parses ---
def extract_yaml(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    blocks = re.findall(r"```(?:ya?ml|YAML)?\s*\n(.*?)```", text, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:ya?ml)?\s*\n(.*)", text, flags=re.S)
    return (m.group(1) if m else text).strip()


def parses(y):
    try:
        yaml.safe_load(y); return "yes"
    except Exception as e:
        return "no: " + str(e).split("\n")[0]


# --- Jenkins: take the Groovy out of the reply, then run the quick structure check ---
def extract_code(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    blocks = re.findall(r"```(?:groovy|Groovy|jenkinsfile|Jenkinsfile)?[ \t]*\n(.*?)```", text, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:groovy)?[ \t]*\n(.*)", text, flags=re.S)
    return (m.group(1) if m else text).strip()


def basic_check(src):
    # strip comments and string literals so braces inside them don't count
    s = re.sub(r"/\*.*?\*/", "", src, flags=re.S)
    s = re.sub(r"'''.*?'''|\"\"\".*?\"\"\"", "''", s, flags=re.S)
    s = re.sub(r"'(?:\\.|[^'\\\n])*'|\"(?:\\.|[^\"\\\n])*\"", "''", s)
    s = re.sub(r"//[^\n]*", "", s)
    stack, pairs = [], {")": "(", "]": "[", "}": "{"}
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack.pop() != pairs[ch]:
                return f"no: unbalanced '{ch}'"
    if stack:
        return f"no: {len(stack)} unclosed bracket(s)"
    body = re.sub(r"^\s*(#!.*\n)?", "", s)
    if not re.match(r"\s*(pipeline\s*\{|node\b)", body):
        return "no: does not start with pipeline { or node"
    return "yes"


# Self-test on tiny examples, so a broken helper is caught before the real run
assert extract_yaml("Sure:\n```yaml\ntrigger:\n- main\n```") == "trigger:\n- main"
assert parses("trigger:\n- main") == "yes"
assert basic_check("pipeline {\n  agent any\n}") == "yes"
assert basic_check("pipeline {\n  agent any\n").startswith("no")
print("helpers ok")

## 6. Generate all 30 files

**What this does:** sends each of the 10 descriptions to each of the 3 models (3 × 10 = 30 files) and saves them.

The work is split into two cells:
- **6a** defines `ask_model`, which sends one description to one model and records how long it took and how many
  tokens it read and wrote.
- **6b** runs every model on every task, one model at a time, and unloads each model afterwards to free the GPU.

**Where each file goes**
```
outputs/
├── azure/<model>/<slug>.yml          the generated YAML
├── azure/<model>/<slug>.raw.txt      the model's full reply, before extraction
├── azure/<model>/<slug>.meta.json    seconds, tokens, why it stopped
└── jenkins/<model>/<slug>.groovy     (and .raw.txt, .meta.json)
```

**Safe to re-run.** If Kaggle disconnects, run the notebook again: files that already exist are loaded
instead of being generated again.

**Errors don't stop the run.** If one request fails, the error and the last lines of Ollama's log are printed,
and the notebook moves on to the next task. Running it again retries only the missing files.

**What you should see:** one line per file with the model, platform, project number, seconds and check result.
`finish_reason` should be `stop`. If it is `length`, the model hit the 2048-token limit and its file is cut off.

### 6a. Send one description to one model

In [ ]:
def ask_model(tag, system, description):
    """Send one description to one Ollama model.
    Returns the full reply text and a dict with timing and token counts."""
    t = time.time()
    r = requests.post(f"{OLLAMA}/api/generate", json={
        "model": tag, "system": system, "prompt": description,
        "stream": False, "options": OPTIONS}, timeout=3600)
    if not r.ok:   # show Ollama's own error message, not just the status code
        raise RuntimeError(f"Ollama HTTP {r.status_code}: {r.text[:500]}")
    j = r.json()
    info = {
        "seconds": round(time.time() - t, 1),        # wall-clock time for this reply
        "prompt_tokens": j.get("prompt_eval_count"),  # tokens the model read
        "output_tokens": j.get("eval_count"),         # tokens the model wrote
        "finish_reason": j.get("done_reason"),        # "stop" = finished; "length" = hit num_predict
    }
    return j["response"], info

### 6b. Run every model on every task

In [ ]:
# The two platforms differ only in prompt, file extension, extractor and check
TASKS = [
    {"platform": "azure",   "items": AZURE_ITEMS,   "system": AZURE_SYSTEM,   "ext": "yml",
     "extract": extract_yaml, "check": parses},
    {"platform": "jenkins", "items": JENKINS_ITEMS, "system": JENKINS_SYSTEM, "ext": "groovy",
     "extract": extract_code, "check": basic_check},
]

results = {"azure": [], "jenkins": []}

errors = []

for name, tag in WORKING_MODELS.items():
    print(f"\n=== {name} ===")
    for task in TASKS:
        folder = f"{WORK}/outputs/{task['platform']}/{name}"
        os.makedirs(folder, exist_ok=True)

        for it in task["items"]:
            path = f"{folder}/{it['slug']}.{task['ext']}"
            meta_path = f"{folder}/{it['slug']}.meta.json"

            if os.path.exists(path) and os.path.exists(meta_path):
                # already generated in an earlier run: reuse it
                generated, info = open(path).read(), json.load(open(meta_path))
            else:
                try:
                    raw, info = ask_model(tag, task["system"], it["description"])
                except Exception as e:
                    # record the error and carry on with the next task
                    print(f"{task['platform']:7s} #{it['no']}  ERROR: {e}")
                    print(ollama_log_tail(10))
                    errors.append({"model": name, "platform": task["platform"], "no": it["no"], "error": str(e)})
                    continue
                generated = task["extract"](raw)
                open(path, "w").write(generated)
                open(f"{folder}/{it['slug']}.raw.txt", "w").write(raw)
                json.dump(info, open(meta_path, "w"), indent=2)

            check = task["check"](generated)
            results[task["platform"]].append({"no": it["no"], "slug": it["slug"], "title": it["title"],
                                              "model": name, "check": check, **info,
                                              "generated": generated})
            print(f"{task['platform']:7s} #{it['no']}  {info['seconds']:6.1f}s  "
                  f"{info['finish_reason'] or '':6s}  check: {check[:50]}")

    # unload the model so the next one has the whole GPU
    requests.post(f"{OLLAMA}/api/generate", json={"model": tag, "keep_alive": 0})

print(f"\nDone: {sum(len(v) for v in results.values())} files, {len(errors)} errors")
if errors:
    print("Run the notebook again to retry only the missing files.")

## 7. Save the results

**What this does:** writes the results as CSV files and zips everything into one download.

| File | One row per | Use it for |
|---|---|---|
| `azure_generated_for_sheet.csv` | project | pasting into the evaluation sheet: one column per model |
| `jenkins_generated_for_sheet.csv` | project | the same, for Jenkins |
| `azure_generated_long.csv` | project × model | analysis: check result, seconds, tokens, finish reason |
| `jenkins_generated_long.csv` | project × model | the same, for Jenkins |
| `free_models_outputs.zip` | | everything in `outputs/` plus the 4 CSVs, `ollama.log` and `errors.csv` (if any), in one download |

**Importing into Google Sheets:** File → Import → Upload, choose a `_for_sheet.csv`, and pick
**Insert new sheet** so the existing tabs are not overwritten. Each file keeps its line breaks inside its cell.

**What you should see:** a summary table per platform and the total generation time per model.

In [ ]:
for platform, rows in results.items():
    long = pd.DataFrame(rows)
    long.to_csv(f"{WORK}/{platform}_generated_long.csv", index=False)

    # one row per project, one column per model
    wide = long.pivot(index=["no", "title"], columns="model", values="generated").reset_index()
    wide = wide.reindex(columns=["no", "title"] + list(MODELS))   # a failed model gets an empty column
    label = "YAML" if platform == "azure" else "Groovy"
    wide.columns = ["No", "Title"] + [f"Generated {label} ({m})" for m in MODELS]
    wide.to_csv(f"{WORK}/{platform}_generated_for_sheet.csv", index=False)

    print(f"\n{platform.upper()}")
    display(long[["no", "model", "check", "seconds", "output_tokens", "finish_reason"]])

# Generation time per model (both platforms together)
times = pd.DataFrame([r for rows in results.values() for r in rows]).groupby("model")["seconds"].agg(["sum", "mean"])
times.columns = ["total seconds", "seconds per file"]
display(times.round(1))

if errors:
    pd.DataFrame(errors).to_csv(f"{WORK}/outputs/errors.csv", index=False)
    print(f"{len(errors)} errors saved to outputs/errors.csv")
elif os.path.exists(f"{WORK}/outputs/errors.csv"):
    os.remove(f"{WORK}/outputs/errors.csv")   # left over from an earlier run
shutil.copy(f"{WORK}/ollama.log", f"{WORK}/outputs/ollama.log")   # Ollama's log, for checking errors later

# Zip outputs/ together with the CSVs
for f in ["azure_generated_long.csv", "azure_generated_for_sheet.csv",
          "jenkins_generated_long.csv", "jenkins_generated_for_sheet.csv"]:
    shutil.copy(f"{WORK}/{f}", f"{WORK}/outputs/{f}")
shutil.make_archive(f"{WORK}/free_models_outputs", "zip", WORK, "outputs")
print("\nSaved the 4 CSVs and free_models_outputs.zip (Output tab)")

## 8. Print every generated file

**What this does:** prints all 30 files, grouped by project, so you can read them or copy them
straight from the notebook page. This is also the backup if the Output tab shows no files.

In [ ]:
for task in TASKS:
    for it in task["items"]:
        print("=" * 100)
        print(f"{task['platform'].upper()} #{it['no']}  {it['title']}")
        for row in results[task["platform"]]:
            if row["no"] == it["no"]:
                print("-" * 100)
                print(f"[{row['model']}]  check: {row['check']}   ({row['seconds']}s)\n")
                print(row["generated"])

## 9. Next step: grading the outputs

This notebook only **generates** files. Each one is then compared by hand with the original file
(linked in `azure/descriptions.md` and `jenkins/descriptions.md`) and given one remark and a comment
saying exactly what differs. Use the **same scale as for the qwen and deepseek models**, so all six can be compared:

| Remark | Meaning |
|---|---|
| **Correct** | matches the original; any differences don't change what the pipeline does |
| **Mostly correct** | works, with only small differences such as a display name or an extra harmless option |
| **Partially correct** | the structure is right, but something is missing or wrong, so part of the pipeline misbehaves |
| **Not correct** | would fail to load or fail when run, or does something unsafe |

**Grade the same failures the same way, every time**
- A pipeline Jenkins or Azure would **refuse to load** (invalid syntax, an option placed as a step) → **Not correct**.
- **Printing any part of a secret** to the logs → no better than **Partially correct**.
- A **wrong path or input name** that makes a job fail → **Not correct**; one the pipeline would ignore safely → **Partially correct**.
- Something **added that the description didn't ask for**, such as a branch filter, is a difference even if
  it looks sensible, and should be mentioned in the comment.
- Groovy **variables in single quotes** (`'${var}'`), which Groovy doesn't fill in → the step that uses them fails.

**Report by scope too:** give the remark counts for the build-and-test configurations (Azure 1–3) and for the
configurations that also deliver or deploy (the other 7). For the qwen and deepseek models, most "Not correct" verdicts came from the
deploy-heavy tasks (Azure 4, Jenkins 5).

**Things to report next to the remarks:** the check result, `finish_reason`, and the seconds per file from Section 7.

## 10. Troubleshooting

| Problem | Likely cause | Fix |
|---|---|---|
| `Ollama did not start` | Internet is off | Settings → Internet **On**, then run Section 1 again |
| A model download stops | network hiccup | run Section 2 again; it resumes and skips finished models |
| `CUDA out of memory` or a very slow model | GPU not on | Settings → Accelerator **GPU T4 x2**, then restart and run all |
| `HTTP 500` from Ollama | the model can't load on this GPU (memory or GPU type) | read the error printed in Section 2b; switch the accelerator (T4 x2 ↔ P100) and Save & Run All again |
| `GPU: NO GPU` in Section 1 | accelerator is off | Settings → Accelerator → GPU, then Save & Run All again |
| `finish_reason` is `length` | the reply hit 2048 tokens | keep the file as it is (the same limit the other models had) and note it in the comment |
| A check says `no: ...` | the model wrote broken code | keep it: that is a result. Grade it like any other file |
| The Output tab shows 0 files | the version was saved with **Quick Save** | use **Save & Run All (Commit)**, or copy the files from Section 8 |
| `ConcurrencyViolation` when saving | the notebook is open in two tabs | close the other tab, reload this one, then save again |
| Kaggle disconnects during the run | session timeout | run all again: files already made are skipped |